<a href="https://colab.research.google.com/github/ShivamTiwari217/Spotify-Song-Recommendation-Engine/blob/main/ANN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
#Importing Necessary Libraries


import pandas as pd
import numpy as np
import joblib
import logging
import time

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors



In [74]:
#Logging Setup

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("SongRecommender")



In [75]:
#Metrics Tracker

METRICS = {
    "total_requests": 0,
    "successful_recommendations": 0,
    "fallback_used": 0,
    "avg_latency_ms": []
}


In [76]:
#Load, Clean, Deduplicate Data

data_path = "spotify_songs.csv"

audio_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo"
]

required_columns = audio_features + [
    "track_name", "track_artist",
    "playlist_genre", "track_popularity"
]

df = pd.read_csv(data_path)[required_columns].dropna()

# Deduplicate songs (playlist duplicates)
df = (
    df.sort_values("track_popularity", ascending=False)
      .drop_duplicates(subset=["track_name", "track_artist"], keep="first")
      .reset_index(drop=True)
)

logger.info(f"Dataset size after deduplication: {df.shape}")


In [77]:
#Random Sampling

SAMPLE_SIZE = 10001

df_sample = (
    df.sample(n=SAMPLE_SIZE, random_state=814) #Random State set to 814 to represent the last digits of each member's enrollment ID
      .reset_index(drop=True)
)

logger.info(f"Sampled dataset size: {df_sample.shape}")


In [78]:
#Feature Scaling

scaler = StandardScaler()
X = scaler.fit_transform(df[audio_features])

# Convert to float32 (important for speed & memory)
X = X.astype("float32")

In [79]:
#Popularity Normalization

df["popularity_norm"] = (
    df["track_popularity"] - df["track_popularity"].min()
) / (
    df["track_popularity"].max() - df["track_popularity"].min() + 1e-5
)

In [80]:
#ANN Model (Cosine Similarity)

nn_model = NearestNeighbors(
    n_neighbors=30,        # search more, rank later
    metric="cosine",
    algorithm="auto"
)

nn_model.fit(X)

print("ANN model trained successfully.")
#

ANN model trained successfully.


In [81]:
#Fallback Logic: Popular Songs

def random_popular_fallback(top_n=10):
    """
    Returns random songs from the top 20% most popular tracks
    """
    popularity_threshold = df_sample["track_popularity"].quantile(0.80)

    popular_pool = df_sample[
        df_sample["track_popularity"] >= popularity_threshold
    ]

    return (
        popular_pool
        .sample(n=min(top_n, len(popular_pool)), random_state=814)
        [["track_name", "track_artist", "playlist_genre", "track_popularity"]]
        .reset_index(drop=True)
    )


In [84]:
#Recommendation Function

def recommend_songs(song_name, artist=None, top_n=10, alpha=0.75):

    # ---- Check if song exists in the sample ----
    if artist:
        matches = df_sample[
            (df_sample["track_name"].str.lower() == song_name.lower()) &
            (df_sample["track_artist"].str.lower() == artist.lower())
        ]
    else:
        matches = df_sample[
            df_sample["track_name"].str.lower() == song_name.lower()
        ]

    # ---- IF SONG DOES NOT EXIST → FALLBACK ----
    if matches.empty:
        return random_popular_fallback(top_n)

    # ---- ELSE → NORMAL RECOMMENDATION PIPELINE ----
    idx = matches.index[0]
    query_vector = X[idx].reshape(1, -1)

    n_neighbors = min(30, len(df_sample))
    distances, indices = nn_model.kneighbors(
        query_vector,
        n_neighbors=n_neighbors
    )

    recommendations = []
    seen_tracks = set()

    for i, dist in zip(indices[0], distances[0]):
        if i == idx or i >= len(df_sample):
            continue

        track_key = (
            df_sample.iloc[i]["track_name"],
            df_sample.iloc[i]["track_artist"]
        )

        if track_key in seen_tracks:
            continue

        seen_tracks.add(track_key)

        similarity = 1 - dist
        popularity = df_sample.iloc[i]["popularity_norm"]
        final_score = alpha * similarity + (1 - alpha) * popularity

        recommendations.append({
            "track_name": df_sample.iloc[i]["track_name"],
            "artist": df_sample.iloc[i]["track_artist"],
            "genre": df_sample.iloc[i]["playlist_genre"],
            "similarity_score": round(similarity, 3),
            "popularity_score": round(popularity, 3),
            "final_score": round(final_score, 3)
        })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [83]:
joblib.dump(nn_model, "song_ann_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(df, "song_metadata.pkl")

print("✅ Model artifacts saved successfully.")

✅ Model artifacts saved successfully.


In [85]:
import os
os.makedirs("models", exist_ok=True)
